# scRNA-seq supplemental analysis — MHC II expression in epithelial cells vs macrophages in LUAD

**Goal:** Compare MHC II and co-stimulatory molecule expression between epithelial
cells and macrophage/monocytes within the same tissue region (primary tumor and
normal adjacent) in LUAD. Shows that MHC II expression in malignant epithelial
cells is comparable in magnitude to professional antigen presenting cells.

**Input:** Salcher et al. pan-cancer scRNA-seq atlas (`luad.h5ad`), subset to
lung adenocarcinoma, primary tumor and normal adjacent tissue.

**Output:** Supplemental figures for figure 2 comparing MHC II and co-stimulatory
molecule expression across cell types and tissue regions.

## Setup

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell analysis
import anndata as ad
import scanpy as sc

# statistics
from scipy.stats import shapiro, wilcoxon, ttest_rel

# utilities
from pathlib import Path
import yaml

# ceiba modules
from ceiba.plot_utils import sig_label

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 20})

# paper-wide color palettes
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
luad_path = repo_root / cfg['outputs']['processed'] / 'luad.h5ad'

# output paths
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

# create output directories
fig_out   = fig_dir / 'figure2-supp'
table_out = table_dir / 'figure2-supp'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# load LUAD object — all cell types, tumor_primary + normal_adjacent
luad = sc.read_h5ad(luad_path)
print(f'Loaded LUAD object: {luad.n_obs:,} cells')
print(f'Cell types:\n{luad.obs["ann_coarse"].value_counts()}')

In [ ]:
# core MHC II antigen presentation genes
mhc2_genes = ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1',
               'HLA-DQA1', 'HLA-DQB1', 'CD74', 'CIITA',
               'HLA-DMA', 'HLA-DMB', 'HLA-DOA', 'HLA-DOB']

# co-stimulatory and immune checkpoint molecules
costim_genes = ['CD80', 'CD86', 'ICAM1', 'CD40', 'ICOSLG',
                'CD58', 'CD274', 'PDCD1LG2', 'PVR', 'NECTIN2',
                'LGALS9', 'CEACAM1', 'TNFRSF14', 'CD44', 'S100P']

def resolve_ensembl(adata, gene_list):
    """
    Map gene symbols to Ensembl IDs via var['feature_name'].

    Parameters
    ----------
    adata : AnnData
    gene_list : list of str
        Gene symbols to resolve.

    Returns
    -------
    dict : {gene_symbol: [ensembl_id, ...]}
    """
    out = {}
    for gene in gene_list:
        hits = list(adata.var.index[adata.var['feature_name'] == gene])
        if not hits:
            print(f'Warning: {gene} not found in var["feature_name"]')
        out[gene] = hits
    return out

mhc2_dict   = resolve_ensembl(luad, mhc2_genes)
costim_dict = resolve_ensembl(luad, costim_genes)

ens_mhc2   = [item for sublist in mhc2_dict.values()   for item in sublist]
ens_costim = [item for sublist in costim_dict.values() for item in sublist]

print(f'MHC II genes resolved:       {len(ens_mhc2)} Ensembl IDs')
print(f'Co-stimulatory genes resolved: {len(ens_costim)} Ensembl IDs')

## Helper functions

Three plotting functions used across the supplemental panels.
All operate on the full LUAD object and perform subsetting internally.
Pending migration to `ceiba.plot_utils`.

In [ ]:
def plot_genes_paired_luad(
    adata,
    genes,
    palette=('tab:grey', 'tab:red'),
    celltype='Epithelial cell',
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Plot paired tumor vs normal adjacent expression for a list of genes.

    Supports any cell type in ann_coarse. For epithelial cells, tumor cells
    are restricted to ann_fine == 'Cancer cells' to exclude non-malignant
    epithelial cells in the tumor_primary compartment. For all other cell
    types, all cells of that type in tumor_primary are included.

    Only donors with both origins represented are included (paired design).
    Statistical test is selected based on test_mode — nonparametric (Wilcoxon)
    is recommended for consistency across genes.

    Parameters
    ----------
    adata : AnnData
        Atlas object. Subsetting to celltype and LUAD is performed internally.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    palette : tuple
        Colors for (normal, tumor).
    celltype : str
        ann_coarse label to subset to (e.g. 'Epithelial cell',
        'Macrophage/Monocyte').
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    sub = adata[
        (adata.obs['ann_coarse'] == celltype) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['origin']   = sub.obs['origin'].astype(str).str.strip().str.lower()
    sub.obs['ann_fine'] = sub.obs['ann_fine'].astype(str).str.strip().str.lower()

    normal_cells = sub[sub.obs['origin'] == 'normal_adjacent'].copy()

    if 'epithelial' in celltype.lower():
        tumor_cells = sub[
            (sub.obs['origin'] == 'tumor_primary') &
            (sub.obs['ann_fine'] == 'cancer cells')
        ].copy()
    else:
        tumor_cells = sub[sub.obs['origin'] == 'tumor_primary'].copy()

    sub = ad.concat([normal_cells, tumor_cells], axis=0, merge='same')

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'origin':   sub.obs['origin'].values,
                gene:       x,
            })
            .groupby(['donor_id', 'origin'], observed=True)[gene]
            .mean()
            .reset_index()
        )

        donor_counts  = df.groupby('donor_id')['origin'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        order = ['normal_adjacent', 'tumor_primary']

        sns.violinplot(
            data=df, x='origin', y=gene, hue='origin',
            order=order, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='origin', y=gene, hue='origin',
            order=order, palette=palette,
            dodge=False, size=6, alpha=0.7,
            ax=ax, legend=False,
        )

        normal_vals, tumor_vals = [], []
        for did, vals in df.groupby('donor_id'):
            norm_val  = vals.loc[vals['origin'] == 'normal_adjacent', gene].values[0]
            tumor_val = vals.loc[vals['origin'] == 'tumor_primary',   gene].values[0]
            ax.plot([0, 1], [norm_val, tumor_val], color='gray', alpha=0.4, linewidth=0.8)
            normal_vals.append(norm_val)
            tumor_vals.append(tumor_val)

        diff   = np.array(tumor_vals) - np.array(normal_vals)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df[gene].max()
        yoff  = (df[gene].max() - df[gene].min()) * 0.15
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom',
                fontsize=28, fontweight='bold')
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(gene)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal\nAdjacent', 'Primary\nTumor'])
        ax.set_xlabel('')
        ax.set_ylabel('Mean expression' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':        gene,
            'n_pairs':     len(diff),
            'Normality_p': p_norm,
            'Test':        test_name,
            'Stat':        stat,
            'p_value':     p,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)


def plot_genes_pct_expressing_luad(
    adata,
    genes,
    palette=('tab:grey', 'tab:red'),
    celltype='Epithelial cell',
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Plot the percent of cells per donor expressing each gene (expression > 0),
    comparing normal adjacent vs primary tumor in paired LUAD donors.

    Complements plot_genes_paired_luad — percent detection is robust to
    zero-inflation and captures presence/absence signal independently of
    mean expression magnitude.

    Supports any cell type in ann_coarse. For epithelial cells, tumor cells
    are restricted to ann_fine == 'Cancer cells'. For all other cell types,
    all cells of that type in tumor_primary are included.

    Parameters
    ----------
    adata : AnnData
        Atlas object. Subsetting to celltype and LUAD is performed internally.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    palette : tuple
        Colors for (normal, tumor).
    celltype : str
        ann_coarse label to subset to.
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    sub = adata[
        (adata.obs['ann_coarse'] == celltype) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['origin']   = sub.obs['origin'].astype(str).str.strip().str.lower()
    sub.obs['ann_fine'] = sub.obs['ann_fine'].astype(str).str.strip().str.lower()

    normal_cells = sub[sub.obs['origin'] == 'normal_adjacent'].copy()

    if 'epithelial' in celltype.lower():
        tumor_cells = sub[
            (sub.obs['origin'] == 'tumor_primary') &
            (sub.obs['ann_fine'] == 'cancer cells')
        ].copy()
    else:
        tumor_cells = sub[sub.obs['origin'] == 'tumor_primary'].copy()

    sub = ad.concat([normal_cells, tumor_cells], axis=0, merge='same')

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'origin':   sub.obs['origin'].values,
                'detected': (x > 0).astype(float),
            })
            .groupby(['donor_id', 'origin'], observed=True)['detected']
            .mean()
            .mul(100.0)
            .reset_index()
            .rename(columns={'detected': 'pct_detected'})
        )

        donor_counts  = df.groupby('donor_id')['origin'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        order = ['normal_adjacent', 'tumor_primary']

        sns.violinplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            dodge=False, size=6, alpha=0.7,
            ax=ax, legend=False,
        )

        normal_vals, tumor_vals = [], []
        for did, vals in df.groupby('donor_id'):
            norm_val  = vals.loc[vals['origin'] == 'normal_adjacent', 'pct_detected'].values[0]
            tumor_val = vals.loc[vals['origin'] == 'tumor_primary',   'pct_detected'].values[0]
            ax.plot([0, 1], [norm_val, tumor_val], color='gray', alpha=0.4, linewidth=0.8)
            normal_vals.append(norm_val)
            tumor_vals.append(tumor_val)

        diff   = np.array(tumor_vals) - np.array(normal_vals)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df['pct_detected'].max()
        yoff  = (df['pct_detected'].max() - df['pct_detected'].min()) * 0.15
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom',
                fontsize=28, fontweight='bold')
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(gene)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal\nAdjacent', 'Primary\nTumor'])
        ax.set_xlabel('')
        ax.set_ylabel('% cells expressing' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':        gene,
            'n_pairs':     len(diff),
            'Normality_p': p_norm,
            'Test':        test_name,
            'Stat':        stat,
            'p_value':     p,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)


def plot_celltype_comparison_luad(
    adata,
    genes,
    tissue='tumor_primary',
    celltypes=('Epithelial cell', 'Macrophage/Monocyte'),
    palette=('#9DD9D2', '#D44D5C'),
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Compare gene expression between two cell types within the same tissue
    region (e.g. epithelial vs macrophage in primary tumor or NAT).

    Donors are paired — only donors with both cell types represented in the
    specified tissue are included. Useful for showing that MHC II expression
    in malignant epithelial cells is comparable to professional APCs.

    Parameters
    ----------
    adata : AnnData
        Atlas object containing LUAD cells.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    tissue : str
        Origin value to filter to ('tumor_primary' or 'normal_adjacent').
    celltypes : tuple of str
        Two ann_coarse labels to compare.
    palette : tuple or str
        Colors for each cell type, or a seaborn palette name.
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    sub = adata[
        (adata.obs['origin'].str.lower() == tissue.lower()) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['ann_coarse'] = sub.obs['ann_coarse'].astype(str).str.strip()

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'celltype': sub.obs['ann_coarse'].values,
                gene:       x,
            })
            .loc[lambda d: d['celltype'].isin(celltypes)]
            .groupby(['donor_id', 'celltype'], observed=True)[gene]
            .mean()
            .reset_index()
        )

        donor_counts  = df.groupby('donor_id')['celltype'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        sns.violinplot(
            data=df, x='celltype', y=gene, hue='celltype',
            order=celltypes, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='celltype', y=gene, hue='celltype',
            order=celltypes, palette=palette,
            dodge=False, size=7, alpha=0.7,
            ax=ax, legend=False,
        )

        vals1, vals2 = [], []
        for did, vals in df.groupby('donor_id'):
            v1 = vals.loc[vals['celltype'] == celltypes[0], gene].values[0]
            v2 = vals.loc[vals['celltype'] == celltypes[1], gene].values[0]
            ax.plot([0, 1], [v1, v2], color='gray', alpha=0.4, linewidth=0.8)
            vals1.append(v1)
            vals2.append(v2)

        diff   = np.array(vals2) - np.array(vals1)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(vals2, vals1)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(vals2, vals1)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(vals2, vals1)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(vals2, vals1)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df[gene].max()
        yoff  = (df[gene].max() - df[gene].min()) * 0.2
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom', fontsize=26)
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(f'{gene} ({tissue})')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(list(celltypes))
        ax.set_xlabel('')
        ax.set_ylabel('Mean expression' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':        gene,
            'tissue':      tissue,
            'n_pairs':     len(diff),
            'Test':        test_name,
            'Stat':        stat,
            'p_value':     p,
            'Normality_p': p_norm,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)

## Supplemental figure — MHC II and co-stimulatory molecule expression in epithelial cells vs macrophages

Two comparisons are made:
1. Within primary tumor — epithelial cells vs macrophage/monocytes
2. Within normal adjacent tissue — epithelial cells vs macrophage/monocytes

MHC II expression in malignant epithelial cells is lower in magnitude than
professional antigen presenting cells, and epithelial cells do not express
classical co-stimulatory molecules such as CD80 and CD86. This distinguishes
epithelial MHC II expression as a separate transcriptional program from
professional antigen presentation.

In [ ]:
# supplemental 2c — epithelial vs macrophage/monocyte in primary tumor
genes = mhc2_genes + costim_genes

stats_tumor = plot_celltype_comparison_luad(
    luad,
    genes=genes,
    tissue='tumor_primary',
    celltypes=('Epithelial cell', 'Macrophage/Monocyte'),
    palette=('tab:blue', 'tab:orange'),
    test_mode='nonparametric',
    nrows=3,
    title='Primary Tumor',
    return_stats=True,
    save_path=fig_out / 'supp_figure2c_celltype_comparison_tumor.pdf',
)
stats_tumor

In [ ]:
# supplemental 2c — epithelial vs macrophage/monocyte in normal adjacent tissue
stats_nat = plot_celltype_comparison_luad(
    luad,
    genes=genes,
    tissue='normal_adjacent',
    celltypes=('Epithelial cell', 'Macrophage/Monocyte'),
    palette=('tab:blue', 'tab:orange'),
    test_mode='nonparametric',
    nrows=3,
    title='Normal Adjacent',
    return_stats=True,
    save_path=fig_out / 'supp_figure2c_celltype_comparison_nat.pdf',
)
stats_nat

## Supplemental figure — MHC II expression in macrophages: tumor vs normal adjacent

Shows that MHC II expression in macrophage/monocytes is not significantly
different between primary tumor and normal adjacent tissue. This rules out
the possibility that the tumor vs NAT difference in epithelial cells is driven
by a general IFN-γ response affecting all cell types equally.

In [ ]:
# supplemental 2d — MHC II expression in macrophages, tumor vs NAT
stats_mac = plot_genes_paired_luad(
    luad,
    genes=mhc2_genes,
    celltype='Macrophage/Monocyte',
    test_mode='nonparametric',
    nrows=3,
    title='Macrophage/Monocytes',
    return_stats=True,
    save_path=fig_out / 'supp_figure2d_mac_mhc2_tumor_vs_nat.pdf',
)
stats_mac

In [ ]:
# supplemental 2d — co-stimulatory molecule expression in macrophages, tumor vs NAT
stats_mac_costim = plot_genes_paired_luad(
    luad,
    genes=costim_genes,
    celltype='Macrophage/Monocyte',
    test_mode='nonparametric',
    nrows=3,
    title='Macrophage/Monocytes — co-stimulatory molecules',
    return_stats=True,
    save_path=fig_out / 'supp_figure2d_mac_costim_tumor_vs_nat.pdf',
)
stats_mac_costim

In [ ]:
# save summary statistics tables
stats_tumor.to_csv(table_out / 'supp_figure2c_celltype_comparison_tumor.csv', index=False)
stats_nat.to_csv(table_out / 'supp_figure2c_celltype_comparison_nat.csv', index=False)
stats_mac.to_csv(table_out / 'supp_figure2d_mac_mhc2.csv', index=False)
stats_mac_costim.to_csv(table_out / 'supp_figure2d_mac_costim.csv', index=False)
print('Tables saved')